In [ ]:
library(ggplot2)
library(dplyr)
library(RColorBrewer)
library(ggh4x)
library(svglite)
library(ragg)

if(!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

cell_lines <- c("A549", "K562")

my_axis <- ggh4x::guide_axis_truncated(
  trunc_lower = unit(0, "npc"),
  trunc_upper = unit(1, "cm")
)

for (cl in cell_lines) {
  
  file_path <- paste0(input_dir, "test_pred_", cl, "_combined_tsne.csv")
  
  data <- read.csv(file_path)
  
  pathway_levels <- sort(unique(data$pathway_level_1)) 
  n_colors <- length(pathway_levels)
  
  getPalette <- colorRampPalette(RColorBrewer::brewer.pal(9, "Set1"))
  my_colors <- getPalette(n_colors)
  names(my_colors) <- pathway_levels 
  
  data_shuffled <- data[sample(nrow(data)), ]
  
  data_types <- c("Ground Truth", "Prediction")
  
  for (dt in data_types) {
    
    plot_data <- data_shuffled %>% filter(Data_Type == dt)
    current_title <- paste0(cl, ": Pathway Distribution - ", dt)
    
    p <- ggplot(plot_data, aes(x = tsne1, y = tsne2, color = pathway_level_1)) +
      ggrastr::rasterise(
        geom_point(size = 0.5, alpha = 0.8),
        dpi = 300,
        dev = "ragg" 
      ) + 
      labs(title = current_title, x = "tSNE 1", y = "tSNE 2") +
      theme_classic() +
      theme(
        plot.margin = margin(t = 2, r = 3, b = 1, l = 3, unit = "cm"),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        axis.line = element_blank(),
        axis.title = element_text(size = 36, face = "bold"),
        plot.title = element_text(size = 40, face = "bold", hjust = 0.5, margin = margin(b = 20)),
        legend.title = element_blank(),
        legend.position = "bottom",
        legend.text = element_text(size = 28),
        legend.spacing.x = unit(3, "cm"),  
        legend.key.width = unit(2.0, "cm"),  
        legend.spacing.y = unit(4, "cm"),  
        legend.key.height = unit(1.2, "cm")  
      ) +
      guides(
        x = my_axis,
        y = my_axis,
        color = guide_legend(
          title.position = "top",
          ncol = 2, 
          byrow = TRUE,  
          override.aes = list(size = 8, alpha = 1)
        )
      ) +
      coord_fixed(ratio = 1) +
      scale_color_manual(values = my_colors, drop = FALSE) 
    
    dt_filename <- gsub(" ", "_", dt)
    base_filename <- paste0(output_dir, "tsne_pathway_", cl, "_", dt_filename)
    plot_width <- 16
    plot_height <- 16
    
    ggsave(filename = paste0(base_filename, ".svg"),
           plot = p, device = svglite, width = plot_width, height = plot_height)
    
    ggsave(filename = paste0(base_filename, ".png"),
           plot = p, width = plot_width, height = plot_height, dpi = 300)
    
    ggsave(filename = paste0(base_filename, ".pdf"),
           plot = p, width = plot_width, height = plot_height)
  }
}
